In [1]:
    import Project2Runner as Runner
    runner = Runner.Project2Runner(n_iter=40)

Find all Shakespeare play files

Found 4 Shakespeare file(s):
  Shakespeare_Macbeth.txt
  Shakespeare_Midsummer_Nights_Dream.txt
  Shakespeare_Much_Ado_About_Nothing.txt
  Shakespeare_Romeo_and_Juliet.txt
Loading spaCy model: en_core_web_md


In [2]:
    runner.run_ner_extraction()


════════════════════════════════════════════════════════════
  STEP 1: NER EXTRACTION
════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────
  Shakespeare_Macbeth.txt
────────────────────────────────────────────────────────────
  Title      : MACBETH
  Characters : 25
  Scenes     : 28
  Entities   : 750 found by default model

  Label summary:
    [CARDINAL    ]  17 unique  e.g. ['One', 'Two', "accus'd", 'eight', 'enough.—Come', 'half']  (+11 more)
    [DATE        ]  21 unique  e.g. ['Days', 'May', 'SECOND', "Thou'lt", 'Tomorrow', 'Tuesday']  (+15 more)
    [FAC         ]   2 unique  e.g. ['Banquo Stick', 'the palace gate']
    [GPE         ]  29 unique  e.g. ['Arabia', 'Birnam', 'England', 'Fife', 'Glamis', "Hear'st"]  (+23 more)
    [LANGUAGE    ]   1 unique  e.g. ['English']
    [LOC         ]   4 unique  e.g. ['Cumberland', 'East', 'Neptune', 'the Western Isles']
    [MONEY       ]   1 unique  e.g. ['Ten thousan

### NER_Extraction

In [1]:
    import spacy
    import src.Project2.NER_Extraction.ner_extraction as NER_Extraction

    print("Loading spaCy model: en_core_web_md ...")
    nlp = spacy.load("en_core_web_md")

    # Discover all Shakespeare play files in train/
    play_files = sorted([
        p for p in NER_Extraction.TRAIN_DIR.rglob("*.txt")
        if "shakespeare" in p.name.lower()
    ])

    if not play_files:
        print(f"ERROR: No Shakespeare files found in {NER_Extraction.TRAIN_DIR}")
        raise SystemExit(1)

    print(f"\nFound {len(play_files)} Shakespeare file(s):")
    for p in play_files:
        print(f"  {p.name}")

    all_records  = []
    all_mislabel = []

    for play_file in play_files:
        print(f"\n{'═'*60}")
        print(f"  {play_file.name}")
        print(f"{'═'*60}")

        root, _    = NER_Extraction.load_play(play_file)
        title      = NER_Extraction.get_title(root)
        characters = NER_Extraction.extract_cast(root)
        scenes     = NER_Extraction.extract_scenes(root)

        print(f"  Title      : {title}")
        print(f"  Characters : {len(characters)}")
        print(f"  Scenes     : {len(scenes)}")

        records, summary = NER_Extraction.run_default_ner(scenes, nlp, title)
        print(f"  Entities   : {len(records):,} found by default model")

        mislabeled = NER_Extraction.analyse_labels(summary, characters, title)
        all_mislabel.extend(mislabeled)

        # Per-play CSV
        NER_Extraction.save_csv(records, NER_Extraction.OUTPUT_DIR / f"01_{play_file.stem}_default_ner.csv")
        all_records.extend(records)

    # Combined CSV
    NER_Extraction.save_csv(all_records, NER_Extraction.OUTPUT_DIR / "01_ALL_default_ner.csv")

    print(f"\n{'═'*60}")
    print(f"  COMPLETE")
    print(f"  Total entity records : {len(all_records):,}")
    print(f"  Total mislabelings   : {len(all_mislabel)}")
    print(f"  Output dir           : {NER_Extraction.OUTPUT_DIR}")
    print(f"\nNext step: run 02_fine_tuning.py")


Loading spaCy model: en_core_web_md ...

Found 4 Shakespeare file(s):
  Shakespeare_Macbeth.txt
  Shakespeare_Midsummer_Nights_Dream.txt
  Shakespeare_Much_Ado_About_Nothing.txt
  Shakespeare_Romeo_and_Juliet.txt

════════════════════════════════════════════════════════════
  Shakespeare_Macbeth.txt
════════════════════════════════════════════════════════════
  Title      : MACBETH
  Characters : 25
  Scenes     : 28
  Entities   : 750 found by default model

  Label summary:
    [CARDINAL    ]  17 unique  e.g. ['One', 'Two', "accus'd", 'eight', 'enough.—Come', 'half']  (+11 more)
    [DATE        ]  21 unique  e.g. ['Days', 'May', 'SECOND', "Thou'lt", 'Tomorrow', 'Tuesday']  (+15 more)
    [FAC         ]   2 unique  e.g. ['Banquo Stick', 'the palace gate']
    [GPE         ]  29 unique  e.g. ['Arabia', 'Birnam', 'England', 'Fife', 'Glamis', "Hear'st"]  (+23 more)
    [LANGUAGE    ]   1 unique  e.g. ['English']
    [LOC         ]   4 unique  e.g. ['Cumberland', 'East', 'Neptune', 'the 

### Fine-Tuning

In [2]:
    import src.Project2.Fine_Tuning.fine_tuning as Fine_Tuning

    play_files = sorted([
        p for p in Fine_Tuning.TRAIN_DIR.rglob("*.txt")
        if "shakespeare" in p.name.lower()
    ])

    if not play_files:
        print(f"ERROR: No Shakespeare files found in { Fine_Tuning.TRAIN_DIR}")
        raise SystemExit(1)

    print(f"Found {len(play_files)} play file(s):")
    for p in play_files:
        print(f"  {p.name}")

    # Build training data — one base nlp just for make_doc
    nlp_base     = spacy.load("en_core_web_md")
    all_examples = []
    play_data    = []    # store (play_file, root, title, characters, scenes) for later

    for play_file in play_files:
        print(f"\n{'─'*60}")
        print(f"  Building training data: {play_file.name}")
        root       =  Fine_Tuning.load_play(play_file)
        title      =  Fine_Tuning.get_title(root)
        characters =  Fine_Tuning.extract_cast(root)
        scenes     =  Fine_Tuning.extract_scenes(root)
        play_data.append((play_file, title, characters, scenes))

        config =  Fine_Tuning.PLAY_CONFIGS.get(play_file.name, {})
        if not config:
            print(f"  WARNING: No config found for {play_file.name} in play_configs.py")
            print(f"           GPE/LOCATION/TITLE labels will be skipped for this play.")

        examples, skipped =  Fine_Tuning.build_training_data_for_play(
            scenes, characters, config, nlp_base
        )
        print(f"  {len(examples):,} training examples  ({skipped} skipped)")
        all_examples.extend(examples)

    print(f"\nTotal training examples across all plays: {len(all_examples):,}")

    # Fine-tune once on all combined examples
    nlp_ft =  Fine_Tuning.fine_tune(all_examples, n_iter=40)

    # Save model
    Fine_Tuning.MODEL_OUT.mkdir(parents=True, exist_ok=True)
    nlp_ft.to_disk( Fine_Tuning.MODEL_OUT)
    print(f"\nFine-tuned model saved → { Fine_Tuning.MODEL_OUT}")

    # Extract and save per-play entities using fine-tuned model
    print("\n=== Extracting entities with fine-tuned model ===")
    all_records = []
    for play_file, title, characters, scenes in play_data:
        records =  Fine_Tuning.extract_and_save(nlp_ft, scenes, title, play_file.stem)
        all_records.extend(records)

    # Combined output
    combined_path =  Fine_Tuning.OUTPUT_DIR / "02_ALL_finetuned_ner.csv"
    combined_path.parent.mkdir(parents=True, exist_ok=True)
    with open(combined_path, "w", newline="", encoding="utf-8") as f:
        writer =  Fine_Tuning.csv.DictWriter(f, fieldnames= Fine_Tuning.ENTITY_FIELDS)
        writer.writeheader()
        writer.writerows(all_records)

    print(f"\n  Combined CSV -> {combined_path.name}")
    print(f"\n{'═'*60}")
    print(f"  COMPLETE — {len(all_records):,} total entity records")
    print(f"\nNext step: run 03_coreference.py")

Found 4 play file(s):
  Shakespeare_Macbeth.txt
  Shakespeare_Midsummer_Nights_Dream.txt
  Shakespeare_Much_Ado_About_Nothing.txt
  Shakespeare_Romeo_and_Juliet.txt

────────────────────────────────────────────────────────────
  Building training data: Shakespeare_Macbeth.txt
  330 training examples  (0 skipped)

────────────────────────────────────────────────────────────
  Building training data: Shakespeare_Midsummer_Nights_Dream.txt


C:\Users\drago\AppData\Local\Programs\Python\Python39\lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "That now Sweno, the Norways' king, craves composit..." with entities "[(20, 26, 'GPE'), (117, 135, 'GPE')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
C:\Users\drago\AppData\Local\Programs\Python\Python39\lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "How far is't call'd to Forres?—What are these, So ..." with entities "[(23, 29, 'GPE')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
C:\Users\drago\AppData\Local\Programs\Python\Python39\lib\site-packages\spacy\training\iob_utils.py:149: User

  425 training examples  (0 skipped)

────────────────────────────────────────────────────────────
  Building training data: Shakespeare_Much_Ado_About_Nothing.txt
  462 training examples  (0 skipped)

────────────────────────────────────────────────────────────
  Building training data: Shakespeare_Romeo_and_Juliet.txt


C:\Users\drago\AppData\Local\Programs\Python\Python39\lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Hark, how they knock!—Who's there?—Romeo, arise, T..." with entities "[(35, 40, 'PERSON')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
C:\Users\drago\AppData\Local\Programs\Python\Python39\lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Then as the manner of our country is, In thy best ..." with entities "[(157, 164, 'PERSON')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
C:\Users\drago\AppData\Local\Programs\Python\Python39\lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W

  557 training examples  (0 skipped)

Total training examples across all plays: 1,774

Loading base model: en_core_web_md ...
Fine-tuning on 1,774 examples for 40 iterations ...
  Iteration  10  NER loss: 96.8027
  Iteration  20  NER loss: 48.2783
  Iteration  30  NER loss: 39.7876
  Iteration  40  NER loss: 28.8268

Fine-tuned model saved → E:\DigiPenMasterCourse\Semester4_2026\cs592_NLP\cs592-natural-language-processing\Chankasemporn_Ju-ve_CS592_NLP_Project\models\shakespeare_ner

=== Extracting entities with fine-tuned model ===


C:\Users\drago\AppData\Local\Programs\Python\Python39\lib\site-packages\spacy\glossary.py:20: UserWarning: [W118] Term 'LOCATION' not found in glossary. It may however be explained in documentation for the corpora used to train the language. Please check `nlp.meta["sources"]` for any relevant links.
  warnings.warn(Warnings.W118.format(term=term))
C:\Users\drago\AppData\Local\Programs\Python\Python39\lib\site-packages\spacy\glossary.py:20: UserWarning: [W118] Term 'TITLE' not found in glossary. It may however be explained in documentation for the corpora used to train the language. Please check `nlp.meta["sources"]` for any relevant links.
  warnings.warn(Warnings.W118.format(term=term))


  [MACBETH]  924 entities  → 02_Shakespeare_Macbeth_finetuned_ner.csv
    GPE         : 16 unique
    LOCATION    : 9 unique
    PERSON      : 78 unique
    TITLE       : 7 unique
  [A MIDSUMMER NIGHT'S DREAM]  984 entities  → 02_Shakespeare_Midsummer_Nights_Dream_finetuned_ner.csv
    GPE         : 5 unique
    LOCATION    : 4 unique
    PERSON      : 84 unique
    TITLE       : 1 unique
  [MUCH ADO ABOUT NOTHING]  1,450 entities  → 02_Shakespeare_Much_Ado_About_Nothing_finetuned_ner.csv
    GPE         : 7 unique
    PERSON      : 53 unique
    TITLE       : 6 unique
  [THE TRAGEDY OF ROMEO AND JULIET]  1,383 entities  → 02_Shakespeare_Romeo_and_Juliet_finetuned_ner.csv
    GPE         : 5 unique
    LOCATION    : 3 unique
    PERSON      : 82 unique
    TITLE       : 7 unique

  Combined CSV → 02_ALL_finetuned_ner.csv

════════════════════════════════════════════════════════════
  COMPLETE — 4,741 total entity records

Next step: run 03_coreference.py


### Coreference

In [3]:
    import src.Project2.Coreference.coreference as Coreference

    print("Loading NER model ...")
    try:
        nlp = spacy.load(str(Coreference.MODEL_PATH))
        print(f"  Fine-tuned model loaded from {Coreference.MODEL_PATH}")
    except Exception:
        print("  Fine-tuned model not found — falling back to en_core_web_md")
        nlp = spacy.load("en_core_web_md")

    play_files = sorted([
        p for p in Coreference.TRAIN_DIR.rglob("*.txt")
        if "shakespeare" in p.name.lower()
    ])

    if not play_files:
        print(f"ERROR: No Shakespeare files found in {Coreference.TRAIN_DIR}")
        raise SystemExit(1)

    print(f"\nFound {len(play_files)} play file(s):")
    for p in play_files:
        print(f"  {p.name}")

    all_records = []

    for play_file in play_files:
        print(f"\n{'═'*60}")
        print(f"  {play_file.name}")
        print(f"{'═'*60}")

        root   = Coreference.load_play(play_file)
        title  = Coreference.get_title(root)
        scenes = Coreference.extract_scenes(root)

        # Load per-play gender config
        config       = Coreference.PLAY_CONFIGS.get(play_file.name, {})
        male_chars   = config.get("male_characters", set())
        female_chars = config.get("female_characters", set())

        if not config:
            print(f"  WARNING: No config in play_configs.py for {play_file.name}")
            print(f"           Pronoun resolution will use fallback (most-recent only)")

        resolver = Coreference.CoreferenceResolver(nlp, male_chars, female_chars)

        play_records = []
        for act_id, scene_id, location, text in scenes:
            resolver.reset()
            records = resolver.resolve_scene(text, act_id, scene_id, title)
            play_records.extend(records)

        print(f"  Title    : {title}")
        print(f"  Scenes   : {len(scenes)}")
        print(f"  Pronouns resolved : {len(play_records):,}")

        # Sample output
        print("  Sample resolutions:")
        for r in play_records[:5]:
            print(f"    Act {r['act']} Sc {r['scene']} | {r['speaker']:20s} | "
                  f"'{r['pronoun']}' → {r['resolved_to']}")

        Coreference.save_csv(play_records, Coreference.OUTPUT_DIR / f"03_{play_file.stem}_coreference.csv")
        all_records.extend(play_records)

    Coreference.save_csv(all_records, Coreference.OUTPUT_DIR / "03_ALL_coreference.csv")

    print(f"\n{'═'*60}")
    print(f"  COMPLETE — {len(all_records):,} total pronoun resolutions")
    print(f"\nNext step: run 04_knowledge_graph.py")


Loading NER model ...
  Fine-tuned model loaded from E:\DigiPenMasterCourse\Semester4_2026\cs592_NLP\cs592-natural-language-processing\Chankasemporn_Ju-ve_CS592_NLP_Project\models\shakespeare_ner

Found 4 play file(s):
  Shakespeare_Macbeth.txt
  Shakespeare_Midsummer_Nights_Dream.txt
  Shakespeare_Much_Ado_About_Nothing.txt
  Shakespeare_Romeo_and_Juliet.txt

════════════════════════════════════════════════════════════
  Shakespeare_Macbeth.txt
════════════════════════════════════════════════════════════
  Title    : MACBETH
  Scenes   : 28
  Pronouns resolved : 766
  Sample resolutions:
    Act 1 Sc 2 | DUNCAN               | 'He' → DUNCAN
    Act 1 Sc 2 | DUNCAN               | 'his' → DUNCAN
    Act 1 Sc 2 | MALCOLM              | 'it' → MALCOLM
    Act 1 Sc 2 | SOLDIER              | 'it' → SOLDIER
    Act 1 Sc 2 | SOLDIER              | 'their' → SOLDIER
  → Saved 766 records  :  03_Shakespeare_Macbeth_coreference.csv

════════════════════════════════════════════════════════════


### Knowledge Graph

In [1]:
    import src.Project2.KnowledgeGraph.knowledge_graph as KnowledgeGraph
    import spacy

    print("=== Loading rules ===")
    rules = KnowledgeGraph.load_rules(KnowledgeGraph.RULES_PATH)

    print("\n=== Loading spaCy model for dependency parsing ===")
    nlp = spacy.load("en_core_web_md")

    print("\n=== Connecting to Memgraph ===")
    mg = KnowledgeGraph.connect_memgraph()

    if mg:
        print("Clearing existing graph data...")
        KnowledgeGraph.clear_old_data(mg)
        print("Graph cleared.")

    play_files = sorted([
        p for p in KnowledgeGraph.TRAIN_DIR.rglob("*.txt")
        if "shakespeare" in p.name.lower()
    ])

    if not play_files:
        print(f"No Shakespeare files found in {KnowledgeGraph.TRAIN_DIR}")
        raise SystemExit(1)

    print(f"\nFound {len(play_files)} play file(s):")
    for p in play_files:
        print(f"  {p.name}")

    total_nodes = 0
    total_rels  = 0

    for play_file in play_files:
        print(f"\n{'='*50}")
        print(f"  Processing: {play_file.name}")
        print(f"{'='*50}")
        nodes, rels  = KnowledgeGraph.populate_play(mg, play_file, rules, nlp)
        total_nodes += nodes
        total_rels  += rels

    print(f"\n{'='*50}")
    print(f"  COMPLETE")
    print(f"  Total nodes   : {total_nodes}")
    print(f"  Total rels    : {total_rels}")
    print(f"\n  Open Memgraph Lab at http://localhost:3000")


=== Loading rules ===

=== Loading spaCy model for dependency parsing ===

=== Connecting to Memgraph ===
Connected to Memgraph successfully.
Clearing existing graph data...
Graph cleared.

Found 4 play file(s):
  Shakespeare_Macbeth.txt
  Shakespeare_Midsummer_Nights_Dream.txt
  Shakespeare_Much_Ado_About_Nothing.txt
  Shakespeare_Romeo_and_Juliet.txt

  Processing: Shakespeare_Macbeth.txt

  Play: MACBETH
  Nodes created/merged : 80
  APPEARS_IN edges     : 48
    Extracted 50 relationships (11 high-confidence)
  Auto-extracted rels  : 50

  Processing: Shakespeare_Midsummer_Nights_Dream.txt

  Play: A MIDSUMMER NIGHT'S DREAM
  Nodes created/merged : 68
  APPEARS_IN edges     : 58
    Extracted 151 relationships (54 high-confidence)
  Auto-extracted rels  : 151

  Processing: Shakespeare_Much_Ado_About_Nothing.txt

  Play: MUCH ADO ABOUT NOTHING
  Nodes created/merged : 44
  APPEARS_IN edges     : 31
    Extracted 85 relationships (36 high-confidence)
  Auto-extracted rels  : 85

  P